# PersonaPlex `<ref>` Context-Injection LoRA Training

Trains (or re-trains) the reference LoRA adapter that teaches PersonaPlex to
correctly consume `<ref>...</ref>` / `<lookup>...</lookup>` injected context,
using the dataset produced by `01_Dataset_Generation.ipynb`.

**Before you run this on real training data, run Section 3 (the contract
check) on its own and read what it prints.** The model-loading code in this
notebook is copied directly from your production `IMTalker/liveTry.py`, so it
is guaranteed to match. The training forward/loss call targets the standard
Moshi-family `LMModel` training contract, but PersonaPlex is NVIDIA's own
checkpoint/fork and could not be verified against its exact source offline --
the contract check runs one real forward+backward pass against your actual
installed model on this pod and tells you immediately whether that
assumption holds, before any GPU time is spent on real training. See the long
comment at the top of `ref_lora_training/common/model_adapter.py` for exactly
what to change if it does not.

**Output:** a LoRA adapter at `<REF_LORA_DIR>/lora/adapter_config.json` +
`adapter_model.safetensors` -- point your `run_imtalker_personaplex.sh`'s
`REF_LORA_DIR` at the parent folder to use it.

## 1. Environment setup

Mirrors `prepare_imtalker_personaplex.sh --with-search`'s pins.

In [ ]:
# %pip install -q "peft>=0.19,<0.20" "transformers==4.52.4" bitsandbytes accelerate sentencepiece huggingface_hub safetensors torch --upgrade
print("Environment cell ready. Uncomment the pip install line above on a fresh pod.")

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "IMTalker").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "IMTalker").exists(), (
    f"Could not find IMTalker/ from {PROJECT_ROOT} -- run from inside the project, "
    "or set PROJECT_ROOT by hand."
)
sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT =", PROJECT_ROOT)

## 2. Configuration

In [ ]:
import torch

# --- Base model: identical to run_imtalker_personaplex.sh's defaults --------
MOSHI_ROOT = os.getenv("MOSHI_ROOT", str(PROJECT_ROOT / "checkpoints" / "personaplex_bnb4"))
MIMI_HF_REPO = "nvidia/personaplex-7b-v1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
QUANTIZE_4BIT = True
NUM_CODEBOOKS = 8          # matches liveTry.py's default; only affects the silent
                           # placeholder audio channels used during training (see
                           # ref_lora_training/common/batching.py's module docstring)
HF_TOKEN = os.getenv("HF_TOKEN", "")   # needs read access to the gated nvidia/personaplex-7b-v1 repo

# --- Dataset ------------------------------------------------------------
DATASET_DIR = PROJECT_ROOT / "ref_lora_training" / "dataset_out"
TRAIN_PATH = DATASET_DIR / "train.jsonl"
VAL_PATH = DATASET_DIR / "val.jsonl"
MAX_SEQ_LEN = 512          # tokens (== frames, see batching.py); truncates from the front

# --- LoRA -----------------------------------------------------------------
LORA_RANK = 16
LORA_ALPHA = None          # None -> 2 * rank, matching IMTalker/generator/train_lora.py's convention
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = None  # None -> auto-discover (see common/model_adapter.discover_target_modules)

# --- Training ---------------------------------------------------------------
OUTPUT_REF_LORA_DIR = PROJECT_ROOT / "ref_lora_training" / "checkpoints_out" / "rag_lora"
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 1e-4
NUM_EPOCHS = 3
LOG_EVERY = 10
EVAL_EVERY = 100
SAVE_EVERY = 200

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print("Config loaded. DEVICE =", DEVICE)

## 3. Load the base model + attach a fresh LoRA

This is the part copied directly from `IMTalker/liveTry.py`'s own model
loading (see `ref_lora_training/common/model_adapter.load_base_model`).

In [ ]:
from ref_lora_training.common.model_adapter import load_base_model, attach_lora

base = load_base_model(
    moshi_root=MOSHI_ROOT, mimi_hf_repo=MIMI_HF_REPO, device=DEVICE,
    quantize_4bit=QUANTIZE_4BIT, num_codebooks=NUM_CODEBOOKS, load_mimi=False,
)
lm, tokenizer = base.lm, base.tokenizer
print("model_type:", base.model_type)
print("base params (B):", sum(p.numel() for p in lm.parameters()) / 1e9)

In [ ]:
peft_model = attach_lora(
    lm, rank=LORA_RANK, alpha=LORA_ALPHA, dropout=LORA_DROPOUT, target_modules=LORA_TARGET_MODULES,
)

## 4. Contract check -- DO NOT SKIP

Runs one real forward+backward pass on a tiny synthetic batch against your
actual loaded model, and tells you which calling convention worked. If this
cell raises, read the error message and `ref_lora_training/common/model_adapter.py`'s
module docstring before doing anything else -- training will not produce a
usable adapter until this passes.

In [ ]:
from ref_lora_training.common.model_adapter import run_contract_check

VOCAB_SIZE_GUESS = getattr(tokenizer, "vocab_size", None) or getattr(tokenizer, "get_piece_size", lambda: 32000)()
FORWARD_ATTEMPT_NAME = run_contract_check(peft_model, NUM_CODEBOOKS, VOCAB_SIZE_GUESS, device=DEVICE)
print("Contract check passed. Using forward convention:", FORWARD_ATTEMPT_NAME)

## 5. Load the dataset and build batches

In [ ]:
from ref_lora_training.common.dataset_builder import read_jsonl

train_episodes = read_jsonl(TRAIN_PATH)
val_episodes = read_jsonl(VAL_PATH)
assert train_episodes, f"no training episodes found at {TRAIN_PATH} -- run 01_Dataset_Generation.ipynb first"
print(f"train episodes: {len(train_episodes)}   val episodes: {len(val_episodes)}")

from collections import Counter
print("train composition:", dict(Counter(e.example_type for e in train_episodes)))

In [ ]:
from ref_lora_training.common.batching import tokenize_episode, resolve_text_pad_id, build_batch

TEXT_PAD_ID = resolve_text_pad_id(tokenizer)

def make_batches(episodes, batch_size, shuffle, seed=0):
    import random
    idxs = list(range(len(episodes)))
    if shuffle:
        random.Random(seed).shuffle(idxs)
    for start in range(0, len(idxs), batch_size):
        chunk = [episodes[i] for i in idxs[start:start + batch_size]]
        tokenized = [tokenize_episode(ep, tokenizer, max_len=MAX_SEQ_LEN) for ep in chunk]
        codes, loss_mask = build_batch(
            tokenized, num_audio_codebooks=NUM_CODEBOOKS, text_pad_id=TEXT_PAD_ID, device=DEVICE,
        )
        yield codes, loss_mask

# quick shape sanity check
_codes, _mask = next(make_batches(train_episodes, BATCH_SIZE, shuffle=True))
print("codes shape:", tuple(_codes.shape), " loss_mask shape:", tuple(_mask.shape))
print("supervised tokens in this batch:", int(_mask.sum().item()), "/", _mask.numel())

## 6. Training loop

Standard QLoRA loop: only the LoRA parameters have `requires_grad=True` (the
4-bit base is frozen), gradient accumulation to reach an effective batch size
of `BATCH_SIZE * GRAD_ACCUM_STEPS`, and periodic val-loss checks + checkpoint
saves in the exact layout `liveTry.py` expects to load.

In [ ]:
from ref_lora_training.common.model_adapter import compute_text_loss, save_adapter
import time

trainable_params = [p for p in peft_model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE)

@torch.no_grad()
def eval_loss(episodes, n_batches=10):
    if not episodes:
        return float("nan")
    peft_model.eval()
    losses = []
    for i, (codes, loss_mask) in enumerate(make_batches(episodes, BATCH_SIZE, shuffle=False)):
        if i >= n_batches:
            break
        loss, _ = compute_text_loss(peft_model, codes, loss_mask, forward_attempt_name=FORWARD_ATTEMPT_NAME)
        losses.append(loss.item())
    peft_model.train()
    return sum(losses) / max(1, len(losses))

peft_model.train()
step = 0
t_start = time.perf_counter()
for epoch in range(NUM_EPOCHS):
    optimizer.zero_grad()
    for micro_step, (codes, loss_mask) in enumerate(make_batches(train_episodes, BATCH_SIZE, shuffle=True, seed=epoch)):
        loss, _ = compute_text_loss(peft_model, codes, loss_mask, forward_attempt_name=FORWARD_ATTEMPT_NAME)
        (loss / GRAD_ACCUM_STEPS).backward()

        if (micro_step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            step += 1

            if step % LOG_EVERY == 0:
                elapsed = time.perf_counter() - t_start
                print(f"epoch {epoch} step {step}: train_loss={loss.item():.4f}  ({elapsed:.0f}s elapsed)")
            if step % EVAL_EVERY == 0:
                v = eval_loss(val_episodes)
                print(f"  -- val_loss={v:.4f} at step {step}")
            if step % SAVE_EVERY == 0:
                save_adapter(peft_model, OUTPUT_REF_LORA_DIR)

save_adapter(peft_model, OUTPUT_REF_LORA_DIR)
print("training complete.")

## 7. Quick qualitative check

Feeds a couple of hand-written `<ref>` blocks through the trained adapter in
plain teacher-forcing mode (not the full streaming avatar pipeline) as a fast
sanity check before wiring it into the live server. This checks the model's
next-token predictions after a `<ref>` block look like real words and not
garbage -- it is NOT a substitute for testing with `ENABLE_SEARCH=1` against
the real pipeline and the scenarios in `logs/detailed_20260908_054904.log`
before calling this done.

In [ ]:
from ref_lora_training.common.ref_format import wrap_with_ref_tags

def quick_check(user_text: str, ref_fact: str, max_new_tokens: int = 30):
    prompt_text = user_text.strip() + "\n" + wrap_with_ref_tags(ref_fact)
    ids = tokenizer.encode(prompt_text)
    print("PROMPT:", prompt_text)
    print("-> (verify this reads like the start of a real, grounded spoken answer,")
    print("    with no literal <ref>/<lookup> text and no leftover markup)")
    # NOTE: this only exercises the text-forward path validated by the contract
    # check above; it does not run the full streaming avatar generation loop.
    # For a true end-to-end check, load this adapter into liveTry.py itself
    # (REF_LORA_DIR=<OUTPUT_REF_LORA_DIR's parent>) and replay the questions
    # from logs/detailed_20260908_054904.log with ENABLE_SEARCH=1.

quick_check("What is Bitcoin trading at right now?", "Bitcoin is trading at $71,204 today.")
quick_check("How is Tesla stock doing today?", "Tesla stock is trading at $309.32 today.")
print("\nReminder: the authoritative test is replaying logs/detailed_20260908_054904.log's")
print("turns against the live server with this adapter loaded.")

## 8. Deploy

Copy (or symlink) the trained adapter to where `run_imtalker_personaplex.sh`
expects it:

```bash
REF_LORA_DIR=/path/to/deployed/rag_lora
mkdir -p "$REF_LORA_DIR"
cp -r ref_lora_training/checkpoints_out/rag_lora/lora "$REF_LORA_DIR/lora"
```

Then launch with `ENABLE_SEARCH=1 REF_LORA_DIR=$REF_LORA_DIR WEB_SEARCH_API_KEY=... ./run_imtalker_personaplex.sh`
and replay the turns from `logs/detailed_20260908_054904.log` by voice, checking
specifically for the three original failure modes: silence after injection,
tag/markup leaking into speech, and a `<ref>` fact bleeding into unrelated
later turns.